In [ ]:
CREATE OR REPLACE PROCEDURE PROC_COPY_BRONZE(
    P_STAGE STRING,
    P_FILE_FORMAT STRING,
    P_TABLES ARRAY  
    -- Example: ARRAY_CONSTRUCT(OBJECT_CONSTRUCT('table_name','BRONZE_CUSTOMERS_DELTA','subfolder','customers'))
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.10'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'main'
AS
$$
from datetime import datetime, timedelta
from snowflake.snowpark import Session

def main(session: Session, P_STAGE: str, P_FILE_FORMAT: str, P_TABLES: list):

    results = []
    success = []
    failed = []

    # Step 1: Folder path (UTC)
    folder_path = (datetime.utcnow() - timedelta(days=3)).strftime("%Y/%b/%d/").title()
    session.sql(f"SELECT '📁 Using folder path: {folder_path}'").collect()

    # Step 2: Loop through each object
    for tbl in P_TABLES:
        table_name = tbl.get("table_name")
        subfolder = tbl.get("subfolder")

        if not table_name or not subfolder:
            failed.append(f"(missing params for entry: {tbl})")
            continue

        # Capitalize first letter for pattern consistency
        pattern_prefix = subfolder.capitalize()

        # Step 3: Build COPY INTO statement
        session.sql(f"TRUNCATE TABLE {table_name}").collect()
        copy_query = f"""
        COPY INTO {table_name}
        FROM @{P_STAGE}/{subfolder}/Delta_load/csv_files/{folder_path}
        FILE_FORMAT = (FORMAT_NAME = {P_FILE_FORMAT})
        MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
        PATTERN = '.*{pattern_prefix}_.*?/part-.*\\\\.parquet';
        """

        try:
            session.sql(copy_query).collect()
            success.append(table_name)
        except Exception as e:
            failed.append(f"{table_name}: {str(e)}")

    # Step 4: Commit or rollback based on outcome
    if len(failed) == 0:
        session.sql("COMMIT").collect()
        summary = f"✅ Loaded {len(success)} tables successfully: {', '.join(success)}"
    else:
        session.sql("ROLLBACK").collect()
        summary = f"⚠️ Partial load. Success: {len(success)}, Failed: {len(failed)} | Errors: {failed}"

    return f"{summary} | Folder path: {folder_path}"
$$;


In [ ]:
CALL PROC_COPY_BRONZE(
    'BRONZE_CSV_DELTA',
    'MY_PARQUET_FORMAT',
    ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT('table_name','BRONZE_CUSTOMERS_DELTA','subfolder','customers'),
        OBJECT_CONSTRUCT('table_name','BRONZE_PRODUCTS_DELTA','subfolder','products'),
        OBJECT_CONSTRUCT('table_name','BRONZE_ORDERS_DELTA','subfolder','orders')
    )
);


In [ ]:
select * from BRONZE_orders_DELTA